In [2]:
import os
import io
import json
import time
from typing import List, Dict, Any
import fitz
from PIL import Image, ImageFile
from dotenv import load_dotenv
from langchain.schema import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

In [19]:
import base64
import io
from PIL import Image
import os
import json
import time
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv

class NotesProcessor:
    def __init__(self, 
                 model='gemini-1.5-flash-8b', 
                 temperature=0, 
                 max_tokens=None, 
                 timeout=None, 
                 max_retries=2):
        """
        Initialize the NotesProcessor with Gemini API configuration.
        """
        load_dotenv()  # Load environment variables
        
        self.llm = ChatGoogleGenerativeAI(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            timeout=timeout,
            max_retries=max_retries
        )
        
        # Predefined category prompts
        self.category_prompts = {
            "Key Terms": "Extract the key terms from the following slides and provide a clear and concise definition for each one.",
            "Problem Types": "Identify the types of problems discussed in the following slides and provide examples if possible.",
            "Formulas": "Extract the key formulas from the following slides, and explain their meaning briefly."
        }

    def extract_pdf_content(self, file_path: str) -> Tuple[str, List[str]]:
        """
        Extract text and images from a PDF file and encode images in Base64.
        """
        pdf_content = ""
        images_base64 = []

        with fitz.open(file_path) as pdf:
            for page in pdf:
                pdf_content += page.get_text()

                for img in page.get_images(full=True):
                    xref = img[0]
                    base_image = pdf.extract_image(xref)
                    image_bytes = base_image["image"]
                    image = Image.open(io.BytesIO(image_bytes))
                    buffered = io.BytesIO()
                    image.save(buffered, format="PNG")
                    images_base64.append(base64.b64encode(buffered.getvalue()).decode("utf-8"))

        print(f"Extracted PDF content: {pdf_content[:100]}")
        print(f"Extracted images: {len(images_base64)} images")

        return pdf_content, images_base64
    
    def extract_text_content(self, file_path: str) -> str:
        """
        Extract text content from a .txt file.
        """
        with open(file_path, "r") as f:
            content = f.read()
            print(f"Extracted text content: {content[:100]}")
            return content

    @staticmethod
    def preprocess_text(text: str) -> str:
        """
        Clean up text by removing extra whitespace and blank lines.
        """
        cleaned_text = "\n".join([line.strip() for line in text.splitlines() if line.strip()])
        print(f"Preprocessed text: {cleaned_text[:100]}")
        return cleaned_text

    def robust_generate(self, messages: List[HumanMessage], retries: int = 5) -> str:
        """
        Robust method for generating text with exponential backoff.
        """
        for attempt in range(retries):
            try:
                response = self.llm.generate([messages])
                print(f"Generated text on attempt {attempt + 1}: {response.generations[0][0].text[:100]}")
                return response.generations[0][0].text
            except Exception as e:
                if "ResourceExhausted" in str(e) and attempt < retries - 1:
                    wait_time = 5 * (attempt + 1)
                    print(f"Resource exhausted. Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    raise RuntimeError(f"Failed after {retries} retries: {e}")

    def process_batch(self, 
                      lecture: List[Tuple[Optional[str], Optional[List[str]]]], 
                      homework: List[Tuple[Optional[str], Optional[List[str]]]],
                      reading: List[Tuple[Optional[str], Optional[List[str]]]],
                      extra: List[Tuple[Optional[str], Optional[List[str]]]],
                      category_prompt: str, 
                      batch_index: int) -> str:
        print(f"Processing batch {batch_index + 1} for: {category_prompt}")
        
        lecture_content = []
        homework_content = []
        reading_content = []
        extra_content = []
                          
        for lecture_data in lecture:
            for text, images in lecture_data:
                if text:
                    lecture_content.append({"type": "text", "text": f"Here is the lecture text:\n{text}"})
                if images:
                    for image_base64 in images:
                        lecture_content.append({"type": "image_url", "image_url": f"data:image/png;base64,{image_base64}"})
                        
        for homework_data in homework:
            for text, images in homework_data:
                if text:
                    homework_content.append({"type": "text", "text": f"Here is the homework text:\n{text}"})
                if images:
                    for image_base64 in images:
                        homework_content.append({"type": "image_url", "image_url": f"data:image/png;base64,{image_base64}"})
                        
                        
        for reading_data in reading:
            for text, images in reading_data:
                if text:
                    reading_content.append({"type": "text", "text": f"Here is the reading text:\n{text}"})
                if images:
                    for image_base64 in images:
                        reading_content.append({"type": "image_url", "image_url": f"data:image/png;base64,{image_base64}"})
                        
                        
        for extra_data in extra:
            for text, images in extra_data:
                if text:
                    extra_content.append({"type": "text", "text": f"Here is the extra text:\n{text}"})
                if images:
                    for image_base64 in images:
                        extra_content.append({"type": "image_url", "image_url": f"data:image/png;base64,{image_base64}"})
        
        lecture_prompt = "Here is the total lecture content. This should be your primary source of truth when generating the content. All other content should be used to answer questions about the lecture."
        homework_prompt = "Here is the total homework content. Only use the content of the assigned homework problems when generating the content."
        reading_prompt = "Here is the total reading content. Only use the content that is relevant to the lecture material when generating the content."
        extra_prompt = "Here is the total extra content. This should be used to answer questions about the lecture material when generating the content."
        
        messages = []
        
        prompt_message = HumanMessage(content=[
            {"type": "text", "text": category_prompt},
        ])
        messages.append(prompt_message)
        if lecture_content:
            lecture_message = HumanMessage(content=[
                {"type": "text", "text": lecture_prompt},
            ] + lecture_content)
            messages.append(lecture_message)
        if homework_content:
            homework_message = HumanMessage(content=[
                {"type": "text", "text": homework_prompt},
            ] + homework_content)
            messages.append(homework_message)
        if reading_content:
            reading_message = HumanMessage(content=[
                {"type": "text", "text": reading_prompt},
            ] + reading_content)
            messages.append(reading_message)
        if extra_content:
            extra_message = HumanMessage(content=[
                {"type": "text", "text": extra_prompt},
            ] + extra_content)
            messages.append(extra_message)

        return self.robust_generate(messages)
    
    
    def generate_concise_summary(self, categorized_results: Dict[str, List[str]]) -> str:
        """
        Generate a concise summary from categorized results.
        """
        final_summary = ""
        for category, results in categorized_results.items():
            combined_results = "\n".join(results)
            summary_prompt = (
                f"Create a concise and cohesive summary of the {category.lower()} from the following data. "
                "Focus on the most important points and avoid redundancy."
            )
            summary_message = HumanMessage(content=[
                {"type": "text", "text": summary_prompt},
                {"type": "text", "text": combined_results},
            ])
            final_summary_response = self.robust_generate(summary_message)
            final_summary += f"\n--- {category} ---\n{final_summary_response}\n"
        
        return final_summary
    
    def process_slides_with_categories(self, 
                                       folder_path: str, 
                                       batch_size: int = 3, 
                                       output_file: str = "processed_notes.json",
                                       custom_categories: Dict[str, str] = None, 
                                       use_lecture: bool = True, 
                                       use_homework: bool = True, 
                                       use_reading: bool = True, 
                                       use_extra: bool = True) -> Dict[str, List[str]]:
        # Use custom categories if provided, otherwise use default
        category_prompts = custom_categories or self.category_prompts

        lectures, homeworks, readings, extras = [], [], [], []
        
        for folder in os.listdir(folder_path):
            try:
                week_folder_path = os.path.join(folder_path, folder)
                if not os.path.isdir(week_folder_path):
                    continue
                    
                for section in os.listdir(week_folder_path):
                    try:
                        section_folder_path = os.path.join(week_folder_path, section)
                        if not os.path.isdir(section_folder_path):
                            continue
                        
                        section_content = []
                        for file_name in sorted(os.listdir(section_folder_path)):
                            file_path = os.path.join(section_folder_path, file_name)
                            if file_name.endswith(".pdf"):
                                text, images = self.extract_pdf_content(file_path)
                                section_content.append((self.preprocess_text(text), images))
                            elif file_name.endswith(".txt"):
                                text = self.extract_text_content(file_path)
                                section_content.append((self.preprocess_text(text), None))
                        
                        if use_lecture and "lecture" in section.lower():
                            lectures.append(section_content)
                        elif use_homework and "homework" in section.lower():
                            homeworks.append(section_content)
                        elif use_reading and "reading" in section.lower():
                            readings.append(section_content)
                        elif use_extra and "extra" in section.lower():
                            extras.append(section_content)
                        
                    except (OSError, IOError) as e:
                        print(f"Error processing section {section}: {e}")
                        continue
                        
            except (OSError, IOError) as e:
                print(f"Error processing week folder {folder}: {e}")
                continue

        categorized_results = {category: [] for category in category_prompts}
        num_batches = max(len(lectures), len(homeworks), len(readings), len(extras))
        for i in range(0, num_batches, batch_size):
            lecture_batch = lectures[i:i + batch_size] if i < len(lectures) else []
            homework_batch = homeworks[i:i + batch_size] if i < len(homeworks) else []
            reading_batch = readings[i:i + batch_size] if i < len(readings) else []
            extra_batch = extras[i:i + batch_size] if i < len(extras) else []
            for category, prompt in category_prompts.items():
                result = self.process_batch(lecture_batch, homework_batch, reading_batch, extra_batch, prompt, i // batch_size)
                categorized_results[category].append(result)

        # Save processed notes
        with open(output_file, "w") as file:
            json.dump(categorized_results, file, indent=4)

        print(f"Processed notes saved to {output_file}")

        return categorized_results

In [20]:
def main():
    """
    Example usage of the NotesProcessor
    """
    processor = NotesProcessor()
    # Custom categories example
    custom_categories = {
        "Key Terms": "Extract the key terms from the following data and provide a clear and concise definition for each one.",
        "Problem Types": "Identify the types of problems discussed in the following data and provide examples if possible.",
        "Algorithm Solutions": "Extract the key algorithm solutions from the following data, and explain their meaning briefly.",
    }

    folder_path = "LinearProgramming"
    
    # Process slides with default or custom categories
    categorized_results = processor.process_slides_with_categories(
        folder_path, 
        batch_size=1, 
        custom_categories=custom_categories,
        use_lecture=True,
        use_homework=False,
        use_reading=False,
        use_extra=False
    )
    
    # Generate summary
    concise_summary = processor.generate_concise_summary(categorized_results)
    print(concise_summary)
    
    # Save results
    processor.save_results(categorized_results)

In [21]:
main()

I0000 00:00:1733154602.219153 6948278 check_gcp_environment_no_op.cc:29] ALTS: Platforms other than Linux and Windows are not supported


Extracted PDF content: CHAPTER 1
Introduction
This book is mostly about a subject called linear programming. Before deﬁning
Extracted images: 1 images
Preprocessed text: CHAPTER 1
Introduction
This book is mostly about a subject called linear programming. Before deﬁning
Extracted PDF content: 1
Introduction
In this short chapter, we shall explain what
is meant by linear programming and
sketc
Extracted images: 20 images
Preprocessed text: 1
Introduction
In this short chapter, we shall explain what
is meant by linear programming and
sketc
Extracted PDF content: 1. What Is It, and What For?
Linear programming, surprisingly, is not directly related to computer p
Extracted images: 6 images
Preprocessed text: 1. What Is It, and What For?
Linear programming, surprisingly, is not directly related to computer p
Extracted PDF content: 2. Examples
Linear programming is a wonderful tool. But in order to use it, one ﬁrst
has to start su
Extracted images: 12 images
Preprocessed text: 2. Examples
Lin

AttributeError: 'NotesProcessor' object has no attribute 'generate_concise_summary'